# Demo pomocnicze: Behavior Trees + Nav2

Ten notebook łączy dwa wątki z poprzednich ćwiczeń: `py_trees` oraz stos nawigacji Nav2. Robot patroluje okolice zamku w Gazebo zawsze wtedy, gdy nie ma aktywnego celu. Kliknięcie przycisku **Jedź do celu** przełącza drzewo na priorytetowy cel nawigacji. Po dojechaniu do celu żądanie celu jest czyszczone i robot automatycznie wraca do patrolu. **Stop** anuluje aktualne zadanie, publikuje zerowe `/cmd_vel` i zatrzymuje robota.

Rdzeń demo jest w pliku `trees_nav.py`. Ten sam plik można uruchomić jako samodzielny test (`python3 trees_nav.py --auto-test`) i jest importowany przez notebook, więc skrypt oraz ćwiczenie używają tej samej implementacji drzewa.

Jeśli przyciski `ipywidgets` nie renderują się w przeglądarce, notebook pokazuje też zapasowe przyciski HTML oraz funkcje `go_to_goal()`, `stop_robot()` i `resume_patrol()`, które można uruchomić ręcznie z komórki.


## Struktura drzewa

Drzewo jest selektorem priorytetów. W każdym tyknięciu sprawdza gałęzie od góry do dołu i uruchamia pierwszą, której warunek jest spełniony.

```text
Castle Demo
|- Stopped?        -> Cancel Navigation
|- Goal requested? -> Navigate To Goal
`- No goal?        -> Patrol Castle
```

Najważniejszy pomysł: patrol jest fallbackiem. Przyciski nie sterują robotem bezpośrednio. Przycisk celu ustawia stan `GOAL`, przycisk stop ustawia `STOPPED`, a brak aktywnego celu oznacza `PATROL`. Dopiero drzewo zachowań wybiera gałąź, a liście drzewa wywołują Nav2 przez `BasicNavigator`.

Priorytet działa tak:

- `STOPPED` jest najwyżej, więc zatrzymanie przerywa wszystko inne.
- `GOAL` jest wyżej niż patrol, więc kliknięcie celu przerywa patrol i wysyła nowy cel Nav2.
- `PATROL` jest zachowaniem domyślnym: jeśli nie ma celu do wykonania, robot jedzie po kolejnych punktach wokół zamku.
- Gdy `NavigateToGoal` zakończy się sukcesem, cel jest uznany za wykonany, stan wraca do `PATROL`, a robot kontynuuje patrol.

In [ ]:
# Uruchom tę komórkę na początku notebooka.
# Dodaje źródła warsztatu do PYTHONPATH i przygotowuje pomocnicze funkcje terminalowe.
import importlib
import math
import os
import shutil
import subprocess
import sys
import threading
import time
import traceback
from dataclasses import dataclass
from enum import Enum
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, IFrame, Javascript, SVG, display

ROS_DISTRO = os.environ.get("ROS_DISTRO", "jazzy")
WORKSPACE = Path("/home/ubuntu/turtlebot3_ws")
SOURCE_DIR = WORKSPACE / "src" / "jupyter_notebooks"
EXERCISES_DIR = SOURCE_DIR / "exercises"

for path in [SOURCE_DIR, EXERCISES_DIR]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))


def bash(command, check=True):
    full = (
        f"source /opt/ros/{ROS_DISTRO}/setup.bash; "
        f"source {WORKSPACE}/install/setup.bash 2>/dev/null || true; "
        f"{command}"
    )
    return subprocess.run(["bash", "-lc", full], text=True, check=check)


def ros_stdout(command, check=False):
    full = (
        f"source /opt/ros/{ROS_DISTRO}/setup.bash; "
        f"source {WORKSPACE}/install/setup.bash 2>/dev/null || true; "
        f"{command}"
    )
    return subprocess.run(
        ["bash", "-lc", full],
        text=True,
        check=check,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )


def wait_for_ros_command(command, description, timeout_sec=45, period_sec=1.0):
    deadline = time.time() + timeout_sec
    last_output = ""
    while time.time() < deadline:
        result = ros_stdout(command, check=False)
        last_output = result.stdout.strip()
        if result.returncode == 0:
            print(f"OK: {description}")
            return True
        time.sleep(period_sec)
    print(f"Nie udało się potwierdzić: {description}")
    if last_output:
        print(last_output[-1000:])
    return False


def print_latest_ros_logs(lines=80):
    script = (
        "latest=$(find ~/.ros/log -maxdepth 2 -name launch.log -type f "
        "-printf '%T@ %p\n' 2>/dev/null | sort -n | tail -1 | cut -d' ' -f2-); "
        f"if [ -n \"$latest\" ]; then echo \"--- $latest ---\"; tail -n {int(lines)} \"$latest\"; fi"
    )
    result = subprocess.run(
        ["bash", "-lc", script],
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout)


def ensure_ros_environment():
    if shutil.which("ros2") is None:
        print("Nie widzę komendy ros2. Ten notebook trzeba uruchomić w kontenerze ROS.")
        return
    print("ROS_DISTRO:", ROS_DISTRO)
    bash("ros2 --help | head -n 1", check=False)


ensure_ros_environment()


In [ ]:
import helper_services
from run_in_term import run_lxterminal


def terminal(command):
    escaped = command.replace("'", "'\\''")
    run_lxterminal(
        f"bash -lc 'source /opt/ros/{ROS_DISTRO}/setup.bash; "
        f"source {WORKSPACE}/install/setup.bash 2>/dev/null || true; {escaped}'"
    )

## 1. Uruchom symulację i Nav2

Pierwsza komórka sprząta stare procesy warsztatowe i uruchamia Gazebo. Kolejne komórki otwierają podgląd VNC, RViz oraz Nav2.

Ważne: poczekaj na komunikat `Gazebo działa...`, zanim przejdziesz dalej.

In [ ]:
helper_services.cleanup_workshop_processes()
terminal("ros2 launch turtlebot3_gazebo turtlebot3_world.launch.py")

if not wait_for_ros_command("ros2 topic list | grep -qx /clock", "Gazebo publikuje /clock", timeout_sec=45):
    print_latest_ros_logs()
    raise RuntimeError(
        "Gazebo nie wystartował poprawnie. Nie uruchamiaj jeszcze RViz/Nav2; "
        "sprawdź powyższy log. Jeśli widzisz błąd GZ_IP albo std::out_of_range, "
        "to problem dotyczy Gazebo Transport w tym kontenerze/hoście, nie drzewa zachowań."
    )

print("Gazebo działa i publikuje /clock.")

In [ ]:
IFrame("http://localhost:6080", width=900, height=520)

In [ ]:
terminal(helper_services.turtlebot3_nav2_command())

amcl_ready = wait_for_ros_command(
    r"ros2 lifecycle get /amcl 2>/dev/null | grep -qx 'active \[3\]'",
    "AMCL jest aktywny i gotowy na pozycję początkową",
    timeout_sec=45,
)
if not amcl_ready:
    print_latest_ros_logs()
    raise RuntimeError("AMCL nie wystartował; Nav2 nie może dostać pozycji początkowej.")

import rclpy
from rclpy.node import Node
from rclpy.parameter import Parameter

try:
    rclpy.init()
except RuntimeError:
    pass

bootstrap_node = Node(
    "castle_nav2_initial_pose_bootstrap",
    parameter_overrides=[Parameter("use_sim_time", value=True)],
)
helper_services.publish_initial_pose(bootstrap_node, repeats=25)
bootstrap_node.destroy_node()

nav2_ready = wait_for_ros_command(
    r"ros2 lifecycle get /controller_server 2>/dev/null | grep -qx 'active \[3\]' && "
    r"ros2 lifecycle get /planner_server 2>/dev/null | grep -qx 'active \[3\]' && "
    r"ros2 lifecycle get /bt_navigator 2>/dev/null | grep -qx 'active \[3\]' && "
    "ros2 action list | grep -qx /navigate_to_pose",
    "Nav2 ma aktywne węzły controller/planner/bt_navigator oraz akcję /navigate_to_pose",
    timeout_sec=60,
)
if not nav2_ready:
    print_latest_ros_logs()
    raise RuntimeError("Nav2 nie wystartował poprawnie. RViz zostanie uruchomiony dopiero po naprawie Nav2.")

terminal(helper_services.turtlebot3_rviz_command())
print("Nav2 działa; RViz został uruchomiony.")

## 2. Przygotuj węzeł ROS i navigator

Publikujemy pozycję początkową robota do AMCL, a potem tworzymy `BasicNavigator`, który będzie wywoływany przez liście drzewa.

`BasicNavigator` jest prostą fasadą na akcje Nav2. W tym ćwiczeniu używamy go do:

- wysyłania kolejnych punktów patrolu przez `goToPose`,
- anulowania bieżącego celu przez `cancelTask`,
- sprawdzania wyniku przez `isTaskComplete()` i `getResult()`.

In [ ]:
import trees_nav

trees_nav = importlib.reload(trees_nav)
DemoMode = trees_nav.DemoMode

# Domyślne wartości można zmienić w następnej komórce przed inicjalizacją Nav2.
PATROL_WAYPOINTS = list(trees_nav.PATROL_WAYPOINTS)
GOAL_X, GOAL_Y, GOAL_YAW = trees_nav.GOAL_POSE

print("Moduł trees_nav załadowany.")


In [ ]:
# Punkty patrolu dookoła zamku. Zmień je, aby pokazać studentom inną trasę.
PATROL_WAYPOINTS = [
    (0.45, 1.91, 0.0),
    (2.53, 0.99, -1.57),
    (1.72, -1.37, 3.14),
    (-0.25, -1.15, 1.57),
]

# Priorytetowy cel po kliknięciu przycisku „Jedź do celu”.
GOAL_X = 0.50
GOAL_Y = -1.20
GOAL_YAW = 0.0

try:
    runner.stop(cancel=navigator.cancelTask)
except NameError:
    pass
except Exception:
    traceback.print_exc()

demo_node, demo = trees_nav.setup_navigation(
    helper_services,
    waypoints=PATROL_WAYPOINTS,
    goal=(GOAL_X, GOAL_Y, GOAL_YAW),
)
navigator = demo.navigator
state = demo.state

print("Nav2 jest aktywne.")
print("Punkty patrolu:", demo.waypoints)
print("Cel priorytetowy:", demo.goal)


## 3. Drzewo zachowań

Logika drzewa jest w pliku `trees_nav.py`, a notebook importuje ją wprost. Dzięki temu najpierw można przetestować zwykły skrypt Pythona, a potem uruchomić dokładnie ten sam kod z interfejsem Jupyter.

W module są trzy rodzaje elementów:

- warunki (`ModeIs`) tylko sprawdzają stan aplikacji,
- akcje (`PatrolCastle`, `NavigateToGoal`, `CancelNavigation`) wykonują pracę,
- liście drzewa nie implementują własnej nawigacji; wywołują Nav2 przez `BasicNavigator`.

Patrol jest fallbackiem: po osiągnięciu celu albo gdy nie ma aktywnego celu, drzewo wraca do patrolu. Lista `PATROL_WAYPOINTS` jest zdefiniowana w module i może być nadpisana w komórce konfiguracyjnej notebooka, więc uruchomienie komórek w poprawnej kolejności nie kończy się błędem `NameError: PATROL_WAYPOINTS is not defined`.


In [ ]:
import py_trees

print("Logika drzewa jest w pliku trees_nav.py i jest współdzielona przez notebook oraz test automatyczny.")
print("Aktualny stan:", trees_nav.status_text(state))


In [ ]:
def make_tree():
    return trees_nav.make_tree(demo)


TREE_SVG = """
<svg width="760" height="330" viewBox="0 0 760 330" xmlns="http://www.w3.org/2000/svg">
  <style>
    .box { fill: #ffffff; stroke: #334155; stroke-width: 2; rx: 8; }
    .selector { fill: #e0f2fe; stroke: #0369a1; }
    .condition { fill: #fef3c7; stroke: #b45309; }
    .action { fill: #dcfce7; stroke: #15803d; }
    .txt { font-family: DejaVu Sans, Arial, sans-serif; font-size: 15px; fill: #0f172a; }
    .small { font-size: 12px; fill: #475569; }
    .line { stroke: #64748b; stroke-width: 2; fill: none; }
  </style>
  <rect class="box selector" x="280" y="18" width="200" height="54"/>
  <text class="txt" x="380" y="42" text-anchor="middle">Selector</text>
  <text class="small" x="380" y="60" text-anchor="middle">wybiera pierwszy sukces</text>

  <path class="line" d="M380 72 V104"/>
  <path class="line" d="M135 104 H625"/>
  <path class="line" d="M135 104 V130 M380 104 V130 M625 104 V130"/>

  <rect class="box condition" x="60" y="130" width="150" height="50"/>
  <text class="txt" x="135" y="160" text-anchor="middle">STOPPED?</text>
  <rect class="box action" x="50" y="210" width="170" height="58"/>
  <text class="txt" x="135" y="235" text-anchor="middle">Anuluj Nav2</text>
  <text class="small" x="135" y="253" text-anchor="middle">robot staje</text>
  <path class="line" d="M135 180 V210"/>

  <rect class="box condition" x="305" y="130" width="150" height="50"/>
  <text class="txt" x="380" y="160" text-anchor="middle">GOAL?</text>
  <rect class="box action" x="295" y="210" width="170" height="58"/>
  <text class="txt" x="380" y="235" text-anchor="middle">Jedź do celu</text>
  <text class="small" x="380" y="253" text-anchor="middle">po sukcesie czyści cel</text>
  <path class="line" d="M380 180 V210"/>

  <rect class="box condition" x="550" y="130" width="150" height="50"/>
  <text class="txt" x="625" y="160" text-anchor="middle">PATROL</text>
  <rect class="box action" x="540" y="210" width="170" height="58"/>
  <text class="txt" x="625" y="235" text-anchor="middle">Patrol zamku</text>
  <text class="small" x="625" y="253" text-anchor="middle">fallback, gdy nie ma celu</text>
  <path class="line" d="M625 180 V210"/>

  <text class="small" x="380" y="315" text-anchor="middle">Priorytet od lewej do prawej: STOP &gt; GOAL &gt; PATROL fallback</text>
</svg>
"""


tree = make_tree()
tree.setup(timeout=15)
display(SVG(TREE_SVG))
print(py_trees.display.unicode_tree(tree.root))


## 4. Sterowanie z notebooka

Najwygodniejsza wersja używa `ipywidgets`. Jeśli w Twojej przeglądarce zamiast przycisków pojawia się tylko tekstowa reprezentacja widgetu, użyj zapasowych przycisków HTML albo ręcznych funkcji:

```python
go_to_goal()      # ustawia aktywny cel; cel ma priorytet nad patrolem
stop_robot()      # anuluje Nav2 i zatrzymuje robota
resume_patrol()   # czyści cel i wraca do patrolu
```

Po dojechaniu do celu `NavigateToGoal` samo ustawia tryb `PATROL`, więc robot nie zostaje w celu na stałe. Patrol jest fallbackiem wtedy, gdy nie ma aktywnego celu.

In [ ]:
status = widgets.HTML()
tree_view = widgets.Output(layout={"border": "1px solid #ddd", "max_height": "260px", "overflow": "auto"})


def status_text():
    return trees_nav.status_text(state)


def refresh_status():
    text = status_text()
    status.value = text.replace("|", "&nbsp;&nbsp;|&nbsp;&nbsp;")
    return text


def go_to_goal():
    text = demo.request_goal()
    refresh_status()
    print(text)
    return text


def stop_robot():
    text = demo.request_stop()
    refresh_status()
    print(text)
    return text


def resume_patrol():
    text = demo.request_patrol()
    refresh_status()
    print(text)
    return text


def request_goal(_):
    go_to_goal()


def request_stop(_):
    stop_robot()


def request_patrol(_):
    resume_patrol()


# Zapasowe przyciski HTML działają w klasycznym Jupyter Notebook bez ipywidgets.
fallback_controls = HTML("""
<div style="display:flex;gap:8px;align-items:center;margin:8px 0 12px 0;font-family:sans-serif">
  <button style="padding:6px 12px;background:#198754;color:white;border:0;border-radius:4px;cursor:pointer"
          onclick="Jupyter.notebook.kernel.execute('go_to_goal()')">Jedź do celu</button>
  <button style="padding:6px 12px;background:#dc3545;color:white;border:0;border-radius:4px;cursor:pointer"
          onclick="Jupyter.notebook.kernel.execute('stop_robot()')">Stop</button>
  <button style="padding:6px 12px;background:#0dcaf0;color:#111;border:0;border-radius:4px;cursor:pointer"
          onclick="Jupyter.notebook.kernel.execute('resume_patrol()')">Wyczyść cel / patrol</button>
  <span style="color:#555">Jeśli widgety poniżej są tylko tekstem, użyj tych przycisków.</span>
</div>
""")

go_button = widgets.Button(description="Jedź do celu", button_style="success", icon="location-arrow")
stop_button = widgets.Button(description="Stop", button_style="danger", icon="stop")
patrol_button = widgets.Button(description="Wyczyść cel / patrol", button_style="info", icon="refresh")

go_button.on_click(request_goal)
stop_button.on_click(request_stop)
patrol_button.on_click(request_patrol)
refresh_status()

display(fallback_controls)
display(widgets.VBox([widgets.HBox([go_button, stop_button, patrol_button]), status, tree_view]))
print("Sterowanie gotowe. Jeśli przyciski ipywidgets nie renderują się, użyj przycisków HTML albo funkcji go_to_goal(), stop_robot(), resume_patrol().")


In [ ]:
def render_tree(tree):
    refresh_status()
    with tree_view:
        tree_view.clear_output(wait=True)
        print(py_trees.display.unicode_tree(tree.root, show_status=True))


try:
    runner.stop(cancel=navigator.cancelTask)
except NameError:
    pass

runner = trees_nav.TreeRunner(tree, period=0.5, on_tick=render_tree)
runner.start()


## 5. Sprawdzenie ruchu i sprzątanie

Oczekiwana sekwencja działania drzewa po uruchomieniu demo:

1. Gdy nie ma aktywnego celu, gałąź `PATROL fallback` wysyła robota na kolejne punkty patrolowe.
2. Kliknięcie **Jedź do celu** ustawia `DemoMode.GOAL`, anuluje ewentualny patrol i uruchamia priorytetową nawigację do `GOAL_POSE`.
3. Po wyniku `TaskResult.SUCCEEDED` cel jest czyszczony, a tryb wraca do `DemoMode.PATROL`, więc robot automatycznie kontynuuje patrol.
4. Kliknięcie **Stop** ustawia `DemoMode.STOPPED`, anuluje aktywne zadanie Nav2 i publikuje zerowe `/cmd_vel`.

Podczas patrolu lub jazdy do celu robot powinien publikować niezerowe komendy prędkości:

```bash
ros2 topic echo --once /cmd_vel
```

Po kilku sekundach powinny zmieniać się współrzędne odometrii:

```bash
ros2 topic echo --once /odom
```

Samodzielny test skryptu można uruchomić po starcie Gazebo i Nav2:

```bash
python3 trees_nav.py --auto-test
```

Test wykonany w kontenerze `ros_fun` na obrazie `ros_fun:native` 2026-06-13 potwierdził scenariusz fallbacku: robot rozpoczął patrol, po żądaniu **Jedź do celu** dojechał do celu `(0.50, -1.20)`, dostał `TaskResult.SUCCEEDED`, wyczyścił cel i sam wrócił do patrolu na kolejny punkt `(2.53, 0.99)`. W ostatnim przebiegu skryptu `trees_nav.py --auto-test` zapisano 2110 próbek `/odom`, 1680 próbek `/cmd_vel`, 1548 niezerowych komend, około 5.45 m drogi i końcowe `/cmd_vel == (0.0, 0.0)`.

Dodatkowo uruchomienie komórek notebooka z tym samym runnerem potwierdziło render kontrolek: przyciski `ipywidgets` zwróciły MIME `application/vnd.jupyter.widget-view+json`, a fallback HTML zawierał **Jedź do celu**, **Stop** i **Wyczyść cel / patrol**. W tym przebiegu robot przejechał około 2.78 m, `/cmd_vel` miał 641 próbek, 589 niezerowych, a po `stop_robot()` ostatnia komenda była zerowa.

Po zakończeniu demo uruchom ostatnią komórkę. Zatrzyma wątek drzewa, anuluje aktywne zadanie Nav2 i posprząta procesy warsztatowe.


In [ ]:
try:
    runner.stop(cancel=navigator.cancelTask)
except NameError:
    pass

try:
    navigator.cancelTask()
except Exception:
    pass

try:
    demo_node.destroy_node()
except Exception:
    pass

helper_services.cleanup_workshop_processes()
rclpy.try_shutdown()
print("Demo zatrzymane.")
